# Tarea del módulo 6

La entrega sera la copia de este notebook resuelta con el nombre del participante en el nombre del archivo

※ Recuerde copiar este notebook y trabajar sobre la Copia

---

## Opcion 1.
Realizar los 3 sigientes ejercicios, documentar paso a paso cuomo es el proceso, imprimir el accuraccy y las imagenes resultantes de cada uno de ellos.


#Clasificador de perros y gatos

Ejercicio 1

Hacer un clasificador de perros y gatos, definimos las imágenes de 180x180x1 y el número de clases es igual a 2, usar una arquitectura tipo lenet (Accuracy debe ser mayor al 70 % para que se tome como completo)

    


In [1]:
# -*- coding: utf-8 -*-
"""Perros y Gatos - CNN en escala de grises"""

import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import zipfile
from PIL import Image

# Configuración para evitar mensajes de advertencia
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print("Versión de TensorFlow:", tf.__version__)
print("Versión de Keras:", keras.__version__)

Versión de TensorFlow: 2.21.0
Versión de Keras: 3.13.2


In [ ]:
# 1. DESCARGA Y EXTRACCIÓN DEL DATASET
print("\n=== DESCARGANDO DATASET ===")
!curl -O https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip

print("\n=== EXTRAYENDO ARCHIVOS ===")
with zipfile.ZipFile('kagglecatsanddogs_5340.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("\n=== LIMPIANDO IMÁGENES CORRUPTAS ===")
def limpiar_imagenes_corruptas():
    """Elimina imágenes corruptas del dataset"""
    num_skipped = 0
    carpetas = ["Cat", "Dog"]

    for carpeta in carpetas:
        ruta_carpeta = os.path.join("PetImages", carpeta)
        if not os.path.exists(ruta_carpeta):
            print(f"¡Carpeta no encontrada: {ruta_carpeta}!")
            continue

        for nombre_archivo in os.listdir(ruta_carpeta):
            ruta_archivo = os.path.join(ruta_carpeta, nombre_archivo)

            try:
                # Intentar abrir la imagen con PIL
                with Image.open(ruta_archivo) as img:
                    img.verify()  # Verificar integridad

                # Verificar que se puede convertir a escala de grises
                with Image.open(ruta_archivo) as img:
                    img_gray = img.convert('L')
                    if img_gray.size != (180, 180):
                        # Redimensionar y guardar para estandarizar
                        img_gray = img_gray.resize((180, 180))
                        img_gray.save(ruta_archivo)

            except Exception as e:
                print(f"Imagen corrupta eliminada: {ruta_archivo}")
                num_skipped += 1
                try:
                    os.remove(ruta_archivo)
                except:
                    pass

    print(f"Total de imágenes eliminadas: {num_skipped}")
    return num_skipped

# Ejecutar limpieza
num_eliminadas = limpiar_imagenes_corruptas()

In [ ]:
# 2. CONFIGURACIÓN DEL DATASET
print("\n=== CONFIGURANDO DATASET ===")
image_size = (180, 180)
batch_size = 128
num_classes = 2
input_shape = (180, 180, 1)  # Escala de grises

# Cargar dataset con imágenes en escala de grises
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    "PetImages",
    validation_split=0.2,
    subset="both",
    seed=666,
    image_size=image_size,
    batch_size=batch_size,
    color_mode="grayscale",  # Importante: carga en escala de grises
    label_mode='int',  # Etiquetas como enteros (0, 1)
    crop_to_aspect_ratio=True
)

# Normalizar las imágenes (opcional pero recomendado)
normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

# Optimizar el rendimiento del dataset
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print(f"\nDataset cargado exitosamente:")
print(f"Tamaño del lote: {batch_size}")
print(f"Shape de entrada: {input_shape}")
print(f"Número de clases: {num_classes}")

In [ ]:
# 3. VISUALIZAR MUESTRAS DEL DATASET
print("\n=== VISUALIZANDO MUESTRAS ===")
plt.figure(figsize=(15, 6))
for images, labels in train_ds.take(1):
    for i in range(10):
        plt.subplot(2, 5, i + 1)
        plt.imshow(images[i].numpy().squeeze(), cmap='gray')
        plt.title(f"{'Perro' if labels[i] == 1 else 'Gato'}")
        plt.axis('off')
plt.suptitle("Muestras del dataset (Escala de grises)")
plt.tight_layout()
plt.show()


In [ ]:
# 4. CREAR EL MODELO CNN
print("\n=== CREANDO MODELO CNN ===")
model = keras.Sequential(
    [
        keras.Input(shape=input_shape, name="entrada"),

        # Primera capa convolucional
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu", padding='same', name="conv1"),
        layers.MaxPooling2D(pool_size=(2, 2), name="pool1"),

        # Segunda capa convolucional
        layers.Conv2D(64, kernel_size=(3, 3), activation="relu", padding='same', name="conv2"),
        layers.MaxPooling2D(pool_size=(2, 2), name="pool2"),

        # Tercera capa convolucional
        layers.Conv2D(128, kernel_size=(3, 3), activation="relu", padding='same', name="conv3"),
        layers.MaxPooling2D(pool_size=(2, 2), name="pool3"),

        # Cuarta capa convolucional
        layers.Conv2D(256, kernel_size=(3, 3), activation="relu", padding='same', name="conv4"),
        layers.MaxPooling2D(pool_size=(2, 2), name="pool4"),

        # Capas fully connected
        layers.Flatten(name="flatten"),
        layers.Dropout(0.5, name="dropout"),
        layers.Dense(2, activation="softmax", name="salida"),
    ],
    name="CNN_Perros_Gatos"
)

# Mostrar resumen del modelo
model.summary()


In [ ]:
# 5. COMPILAR EL MODELO

# Usamos sparse_categorical_crossentropy porque las etiquetas son enteros
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

print("\n=== MODELO COMPILADO ===")

In [ ]:
# 6. ENTRENAR EL MODELO
epochs = 20

# Guardamos el resultado en la variable 'history' para poder graficar después
history = model.fit(train_ds, epochs=epochs, validation_data=val_ds)

In [ ]:
# 7. EVALUAR EL MODELO
print("\n=== EVALUANDO MODELO ===")
results = model.evaluate(val_ds, verbose=1)
print(f"\nResultados en validación:")
print(f"  - Pérdida: {results[0]*100:.2f}%")
print(f"  - Precisión: {results[1]*100:.2f}%")

In [ ]:
# 8. VISUALIZAR RESULTADOS DEL ENTRENAMIENTO
print("\n=== VISUALIZANDO RESULTADOS ===")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico de precisión
ax1.plot(history.history['accuracy'], label='Entrenamiento', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validación', linewidth=2)
ax1.set_title('Precisión del modelo', fontsize=14)
ax1.set_xlabel('Época')
ax1.set_ylabel('Precisión')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gráfico de pérdida
ax2.plot(history.history['loss'], label='Entrenamiento', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validación', linewidth=2)
ax2.set_title('Pérdida del modelo', fontsize=14)
ax2.set_xlabel('Época')
ax2.set_ylabel('Pérdida')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Resultados del entrenamiento - {epochs} épocas', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 9. PREDICCIONES DE EJEMPLO

# Cargar la imagen específicamente en escala de grises para coincidir con el modelo
img = keras.utils.load_img("PetImages/Cat/666.jpg", target_size=image_size, color_mode="grayscale")
plt.imshow(img, cmap='gray')

img_array = keras.utils.img_to_array(img)
img_array = img_array / 255.0  # Aplicar el mismo reescalado que en el entrenamiento
img_array = keras.ops.expand_dims(img_array, 0)  # Crear eje de lote (batch axis)

predictions = model.predict(img_array)
# Como usamos softmax y sparse_categorical_crossentropy, obtenemos la probabilidad de cada clase
score = predictions[0]
clase_pred = np.argmax(score)
confianza = score[clase_pred]

print(f"Esta imagen es {100 * score[0]:.2f}% Gato y {100 * score[1]:.2f}% Perro.")
print(f"Predicción final: {'Perro' if clase_pred == 1 else 'Gato'} ({100*confianza:.2f}% de confianza)")

In [ ]:
# 10. GUARDAR EL MODELO FINAL
print("\n=== GUARDANDO MODELO ===")
model.save('modelo_perros_gatos_final.keras')
print("Modelo guardado como 'modelo_perros_gatos_final.keras'")

In [ ]:
# 11. ESTADÍSTICAS FINALES
print("\n=== ESTADÍSTICAS FINALES ===")
mejor_accuracy = max(history.history['val_accuracy'])
mejor_epoca = history.history['val_accuracy'].index(mejor_accuracy) + 1
print(f"Mejor precisión en validación: {mejor_accuracy:.4f} (época {mejor_epoca})")
print(f"Precisión final en entrenamiento: {history.history['accuracy'][-1]:.4f}")
print(f"Precisión final en validación: {history.history['val_accuracy'][-1]:.4f}")
print(f"Pérdida final en entrenamiento: {history.history['loss'][-1]:.4f}")
print(f"Pérdida final en validación: {history.history['val_loss'][-1]:.4f}")

print("\n=== ENTRENAMIENTO COMPLETADO ===")

#Perros y gatos con VGG16

---

Ejercicio 2

Crea una red que clasifique el dataset stanford_dogs usando el extractor del VGG16

1- Definir un modelo que tenga el VGG16 como entrada

2- Aplanar el vector de rasgos

3- Definir una o varias capas densas con 120 clases de salida

4- Entrenar el modelo

In [2]:
import tarfile
import os

# 1. DESCARGA Y EXTRACCIÓN DEL DATASET
print("\n=== DESCARGANDO DATASET ===")
!curl -O http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar

print("\n=== EXTRAYENDO ARCHIVOS ===")
# Definir el directorio de salida
output_dir = 'Stanford_Dogs'
# Crear el directorio si no existe
os.makedirs(output_dir, exist_ok=True)
with tarfile.open('images.tar', 'r') as tar_ref:
    tar_ref.extractall(output_dir)

# El dataset de Stanford Dogs tiene una estructura diferente y no requiere
# la limpieza de imágenes corruptas de la misma manera que el dataset de perros y gatos.
# Además, VGG16 espera imágenes RGB, no en escala de grises.
print("\n=== LIMPIEZA DE IMÁGENES CORRUPTAS (NO APLICABLE/MODIFICADO PARA STANFORD DOGS) ===")
print("La limpieza de imágenes corruptas para este dataset se maneja durante la carga del dataset si es necesario.")
# La función `limpiar_imagenes_corruptas` del ejercicio anterior no es compatible con este dataset
# Se ha eliminado para evitar errores y confusiones.


=== DESCARGANDO DATASET ===


  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
  0 756.8M   0 526.4k   0      0 520.3k      0   24:49   00:01   24:48 522.3k
  1 756.8M   1  8.61M   0      0  4.27M      0   02:56   00:02   02:54  4.28M
  3 756.8M   3 25.26M   0      0  8.37M      0   01:30   00:03   01:27  8.38M
  4 756.8M   4 37.57M   0      0  9.33M      0   01:21   00:04   01:17  9.34M
  6 756.8M   6 48.42M   0      0  9.62M      0   01:18   00:05   01:13  9.62M
  7 756.8M   7 57.17M   0      0  9.47M      0   01:19   00:06   01:13 11.27M
  8 756.8M   8 65.72M   0      0  9.33M      0   01:21   00:07   01:14 11.36M
  9 756.8M   9 74.88M   0      0  9.31M      0   01:21   00:08   01:13  9.88M
 11 756.8M  11 83.27M   0      0  9.19M      0   01:22   00:09   01:13  9.08M
 11 756.8M  11 88.40M   0      0  8.79M      0   01:26   00:10 


=== EXTRAYENDO ARCHIVOS ===

=== LIMPIEZA DE IMÁGENES CORRUPTAS (NO APLICABLE/MODIFICADO PARA STANFORD DOGS) ===
La limpieza de imágenes corruptas para este dataset se maneja durante la carga del dataset si es necesario.


In [3]:
import tensorflow as tf
from tensorflow import keras

# 1. CONFIGURACIÓN DEL DATASET PARA VGG16 (RGB)
print("\n=== CONFIGURANDO DATASET (RGB para VGG16) ===")
# VGG16 espera (224, 224, 3) por defecto. Reducimos el tamaño para ahorrar RAM.
image_size_vgg = (150, 150) # Reducido de 224x224 a 150x150
batch_size = 128 # Reducido de 128 a 64 para ahorrar RAM

train_ds_raw, val_ds_raw = keras.utils.image_dataset_from_directory(
    "Stanford_Dogs",
    validation_split=0.2,
    subset="both",
    seed=666,
    image_size=image_size_vgg,
    batch_size=batch_size,
    color_mode="rgb",
    label_mode='int',
    crop_to_aspect_ratio=True
)

# EXTRAER NOMBRES DE CLASES ANTES DE TRANSFORMAR
class_names = train_ds_raw.class_names

# Preprocesamiento específico de VGG16
def preprocess_vgg(x, y):
    x = keras.applications.vgg16.preprocess_input(x)
    return x, y

# Aplicar transformaciones
train_ds_rgb = train_ds_raw.map(preprocess_vgg).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds_rgb = val_ds_raw.map(preprocess_vgg).prefetch(buffer_size=tf.data.AUTOTUNE)


=== CONFIGURANDO DATASET (RGB para VGG16) ===
Found 20580 files belonging to 120 classes.
Using 16464 files for training.
Using 4116 files for validation.


In [4]:
# 2. CARGAR VGG16 Y CREAR EL MODELO
print("\n=== CREANDO MODELO CON VGG16 ===")

base_model = keras.applications.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(150, 150, 3)
)

base_model.trainable = False

model_vgg = keras.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(510, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(120, activation='softmax') # Ajustado a 120 clases para Stanford Dogs
])

model_vgg.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_vgg.summary()


=== CREANDO MODELO CON VGG16 ===
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 510)            │     4,178,430 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 510)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       130,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 120)            │        30,840 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,054,774 (72.69 MB)

 Trainable params: 4,340,086 (16.56 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [6]:
# 3. ENTRENAR EL MODELO VGG16
print("\n=== ENTRENANDO TRANSFER LEARNING ===")
epochs_vgg = 2

# Entrenamos solo las capas densas superiores
model_vgg.fit(
    train_ds_rgb,
    epochs=epochs_vgg,
    validation_data=val_ds_rgb
)


=== ENTRENANDO TRANSFER LEARNING ===
Epoch 1/2
129/129 ━━━━━━━━━━━━━━━━━━━━ 884s 7s/step - accuracy: 0.0098 - loss: 9.0051 - val_accuracy: 0.0112 - val_loss: 4.7865
Epoch 2/2
129/129 ━━━━━━━━━━━━━━━━━━━━ 843s 7s/step - accuracy: 0.0106 - loss: 4.8402 - val_accuracy: 0.0114 - val_loss: 4.7799


In [7]:
# 4. EVALUAR Y COMPARAR
val_loss, val_acc = model_vgg.evaluate(val_ds_rgb)
print(f"\nPrecisión final con VGG16: {val_acc*100:.2f}%")

33/33 ━━━━━━━━━━━━━━━━━━━━ 176s 5s/step - accuracy: 0.0114 - loss: 4.7799

Precisión final con VGG16: 1.14%


In [10]:
# 5. PREDICCIONES DE EJEMPLO CON VGG16
import numpy as np
import matplotlib.pyplot as plt

# Cargar la imagen en RGB
img_path = "D:\Salvador\VSCode\Diplomado_IA\Módulo_VI\Stanford_Dogs\n02085782-Japanese_spaniel\n02085782_38.jpg"
img = keras.utils.load_img(img_path, target_size=image_size_vgg)
plt.imshow(img)
plt.axis('off')

img_array = keras.utils.img_to_array(img)
img_array = np.expand_dims(img_array, 0)

# Preprocesamiento VGG16
img_preprocessed = keras.applications.vgg16.preprocess_input(img_array)

# Predicción
predictions = model_vgg.predict(img_preprocessed)
score = predictions[0]
clase_pred_idx = np.argmax(score)
# 'class_names' ahora existe como variable global definida en el paso anterior
label_pred = class_names[clase_pred_idx]
confianza = score[clase_pred_idx]

print(f"Predicción final: {label_pred} ({100*confianza:.2f}% de confianza)")

<>:6: SyntaxWarning: invalid escape sequence '\S'
<>:6: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Salvador\AppData\Local\Temp\ipykernel_25584\2978124032.py:6: SyntaxWarning: invalid escape sequence '\S'
  img_path = "D:\Salvador\VSCode\Diplomado_IA\Módulo_VI\Stanford_Dogs\n02085782-Japanese_spaniel\n02085782_38.jpg"
C:\Users\Salvador\AppData\Local\Temp\ipykernel_25584\2978124032.py:6: SyntaxWarning: invalid escape sequence '\S'
  img_path = "D:\Salvador\VSCode\Diplomado_IA\Módulo_VI\Stanford_Dogs\n02085782-Japanese_spaniel\n02085782_38.jpg"


OSError: [Errno 22] Invalid argument: 'D:\\Salvador\\VSCode\\Diplomado_IA\\Módulo_VI\\Stanford_Dogs\n02085782-Japanese_spaniel\n02085782_38.jpg'

---

# YOLO - You Only Look Once

Ejercicio YOLO
- Hacer un dataset de 10 imágenes con 2 clases
- Entrenar el modelo usando el dataset ya sea con la implementación de keras o la de ultralytics, se recomienda ultralytics

In [ ]:
!pip install ultralytics

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YiVQil9AfPrlIUDfW5wM")
project = rf.workspace("salvadors-workspace-2wukb").project("animals-ij5d2-bzafq")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:
from ultralytics import YOLO

# Cargar el modelo base para transferencia de aprendizaje
model = YOLO('yolov8n.pt')

# Entrenar usando el dataset de Roboflow
# Nota: Usamos la ruta al archivo data.yaml generado por Roboflow
results = model.train(
    data='/content/animals-1/data.yaml',
    epochs=100
)

print("Entrenamiento finalizado. Los pesos se guardaron en runs/detect/train/weights/best.pt")

In [ ]:
from ultralytics.utils.plotting import Annotator
from PIL import Image
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Definir la ruta de la imagen
image_path = "/content/Saskia2.jpeg"

# Cargar la imagen y reajustar a 640x640
img_original = cv2.imread(image_path)
#img_resized = cv2.resize(img_original, (640, 640))

# Realizar la predicción sobre la imagen reajustada
# Nota: YOLO redimensiona internamente para inferencia, pero esto asegura la consistencia
results = model.predict(source=img_original, conf=0.25)

# Preparar el Annotator con la imagen de 640x640
annotator = Annotator(img_original)

# Dibujar los resultados
for r in results:
    for box in r.boxes:
        annotator.box_label(box.xyxy[0], model.names[int(box.cls[0])])

# Mostrar el resultado final
image_result = annotator.result()
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(image_result, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title("Detección en ejemplo)")
plt.show()